In [1]:
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans

import pandas as pd
import joblib
import mlflow

from mibici_mlops.config import P3_PERFILES_PATH, RANDOM_STATE, ESTACIONES_PATH

/Users/timedina/Documents/timj/Docs/Estudios/MNA/MLOps/Activities/05 - MiBici + MLFlow/code/MiBici_MLOps/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-25 01:49:13.624 | INFO     | mibici_mlops.config:<module>:16 - PROJ_ROOT path is: /Users/timedina/Documents/timj/Docs/Estudios/MNA/MLOps/Activities/05 - MiBici + MLFlow/code/MiBici_MLOps


In [2]:
perfiles = pd.read_parquet(P3_PERFILES_PATH)
estaciones = pd.read_csv(ESTACIONES_PATH)
station_ids = perfiles['estacion'].to_numpy()
parametros = {
    'n_clusters': 3,
    'n_init': 10,
    'random_state': RANDOM_STATE
}
perfiles.columns

Index(['estacion', 'sal_lun_madrugada', 'sal_lun_manana', 'sal_lun_tarde',
       'sal_lun_noche', 'sal_mar_madrugada', 'sal_mar_manana', 'sal_mar_tarde',
       'sal_mar_noche', 'sal_mie_madrugada', 'sal_mie_manana', 'sal_mie_tarde',
       'sal_mie_noche', 'sal_jue_madrugada', 'sal_jue_manana', 'sal_jue_tarde',
       'sal_jue_noche', 'sal_vie_madrugada', 'sal_vie_manana', 'sal_vie_tarde',
       'sal_vie_noche', 'sal_sab_madrugada', 'sal_sab_manana', 'sal_sab_tarde',
       'sal_sab_noche', 'sal_dom_madrugada', 'sal_dom_manana', 'sal_dom_tarde',
       'sal_dom_noche', 'lle_lun_madrugada', 'lle_lun_manana', 'lle_lun_tarde',
       'lle_lun_noche', 'lle_mar_madrugada', 'lle_mar_manana', 'lle_mar_tarde',
       'lle_mar_noche', 'lle_mie_madrugada', 'lle_mie_manana', 'lle_mie_tarde',
       'lle_mie_noche', 'lle_jue_madrugada', 'lle_jue_manana', 'lle_jue_tarde',
       'lle_jue_noche', 'lle_vie_madrugada', 'lle_vie_manana', 'lle_vie_tarde',
       'lle_vie_noche', 'lle_sab_madrugada', 

In [3]:
perfiles = perfiles.copy()
estaciones = estaciones[['id', 'latitude', 'longitude']].rename(columns={
    'id': 'estacion',
    'latitude': 'estacion_lat',
    'longitude': 'estacion_lon',
})
parametros = dict(parametros)

ids_estaciones_train = perfiles[perfiles.conjunto=='entrenamiento'].estacion.to_numpy()
ids_estaciones_test = perfiles[perfiles.conjunto=='prueba'].estacion.to_numpy()

conjunto = perfiles.conjunto
columnas_perfil = [c for c in perfiles.columns if c.startswith(('sal_', 'lle_'))]
X_perfiles = perfiles[columnas_perfil].astype('float64')

X = X_perfiles.loc[conjunto == 'entrenamiento']
X_test = X_perfiles.loc[conjunto == 'prueba']

modelo = KMeans(**parametros)
modelo_test = KMeans(**parametros)

In [4]:
etiquetas = modelo.fit_predict(X)
etiquetas_test = modelo_test.fit_predict(X_test)

asignaciones = pd.concat(
    [
        pd.DataFrame({
            'estacion': ids_estaciones_train,
            'conjunto': 'entrenamiento',
            'cluster': etiquetas,
        }),
        pd.DataFrame({
            'estacion': ids_estaciones_test,
            'conjunto': 'prueba',
            'cluster': etiquetas_test,
        }),
    ],
    ignore_index=True,
)

# Añadir coordenadas a las asignaciones
asignaciones = asignaciones.merge(
    estaciones,
    on='estacion',
    how='left',
    validate='many_to_one',
)

n_clusters = len(set(etiquetas))
if not 1 < n_clusters < len(X):
    raise ValueError(
        'Silhouette requiere al menos 2 clusters y menos clusters que filas.'
    )

grupos_train = pd.Series(
    etiquetas,
    index=ids_estaciones_train,
)
grupos_test = pd.Series(
    etiquetas_test,
    index=ids_estaciones_test,
)

estaciones_comunes = grupos_train.index.intersection(grupos_test.index)
if len(estaciones_comunes) < 2:
    raise ValueError('Se necesitan al menos dos estaciones comunes para calcular ARI.')
ari =  adjusted_rand_score(
    grupos_train.loc[estaciones_comunes],
    grupos_test.loc[estaciones_comunes],
)

metricas = {
    'silhouette': float(silhouette_score(X, etiquetas)),
    'inertia': float(modelo.inertia_),
    'ari_estabilidad': float(ari)
}
metricas

{'silhouette': 0.30304459219089824,
 'inertia': 1.6238219960451903,
 'ari_estabilidad': 0.698469841014881}

## 2. Plot

In [5]:
from mibici_mlops.config import P3_CLUSTERS_PATH, P3_PERFILES_PATH
import folium

In [6]:
COLORES = ['red', 'blue', 'green', 'purple', 'orange', 'yellow', 'brown', 'gray', 'pink', 'black']
clusters = pd.read_parquet(P3_CLUSTERS_PATH)
perfiles = pd.read_parquet(P3_PERFILES_PATH)
print(clusters.head(3))
clusters.shape

   estacion       conjunto  cluster  estacion_lat  estacion_lon
0        80  entrenamiento        2     20.668880   -103.348460
1        39  entrenamiento        2     20.678034   -103.358466
2       323  entrenamiento        2     20.704670   -103.345589


(800, 5)

In [7]:
asignaciones = clusters[clusters.conjunto == 'prueba']
perfiles = perfiles[perfiles.conjunto == 'prueba']

perfiles_con_cluster = perfiles.merge(
    asignaciones[['estacion', 'conjunto', 'cluster']],
    on=['estacion', 'conjunto'],
    validate='one_to_one',
)

In [ ]:
mapa = folium.Map(
    location=[asignaciones.estacion_lat.mean(), asignaciones.estacion_lon.mean()],
    zoom_start=14,
    tiles='OpenStreetMap',
)

grupos = sorted(asignaciones.cluster.dropna().unique())

for i, grupo in enumerate(grupos):
    color = COLORES[i % len(COLORES)]
    capa = folium.FeatureGroup(name=f'Grupo {grupo}')
    puntos = asignaciones[asignaciones['cluster'] == grupo]

    for _, fila in puntos.iterrows():
        folium.CircleMarker(
            location=[fila.estacion_lat, fila.estacion_lon],
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            tooltip=f'Estación {fila.estacion} · Grupo {grupo}',
        ).add_to(capa)

    capa.add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa

In [10]:
grupos = sorted(perfiles_con_cluster["cluster"].dropna().unique())
colores = {
    grupo: COLORES[i % len(COLORES)]
    for i, grupo in enumerate(grupos)
}

columnas_sal = [c for c in perfiles_con_cluster if c.startswith("sal_")]
columnas_lle = [c for c in perfiles_con_cluster if c.startswith("lle_")]

promedios_sal = perfiles_con_cluster.groupby("cluster")[columnas_sal].mean()
promedios_lle = perfiles_con_cluster.groupby("cluster")[columnas_lle].mean()

for grupo in promedios_sal.index:
    salidas = ", ".join(
        f"{c.removeprefix('sal_')} ({v:.1%})"
        for c, v in promedios_sal.loc[grupo].nlargest(3).items()
    )
    llegadas = ", ".join(
        f"{c.removeprefix('lle_')} ({v:.1%})"
        for c, v in promedios_lle.loc[grupo].nlargest(3).items()
    )
    print(f"Grupo {grupo} ({colores[grupo]}) — salidas: {salidas}; llegadas: {llegadas}")

Grupo 0 (red) — salidas: mar_manana (6.3%), mie_manana (6.1%), jue_manana (5.9%); llegadas: mie_noche (6.5%), mar_noche (6.4%), lun_noche (6.1%)
Grupo 1 (blue) — salidas: mar_tarde (7.1%), mie_tarde (7.1%), lun_tarde (6.7%); llegadas: mar_manana (6.8%), mie_manana (6.6%), jue_manana (6.3%)
Grupo 2 (green) — salidas: dom_manana (13.6%), dom_tarde (9.5%), mie_tarde (5.1%); llegadas: dom_manana (15.2%), dom_tarde (11.1%), sab_noche (4.9%)
